In [1]:
%cd '/home/mojtaba.kananisarche/'

import pandas as pd
import dataframe_image as dfi
from Zh.src.utils.visualize_results import confusion_matrix_results, table_results

/home/mojtaba.kananisarche


In [ ]:
# path = "/home/mojtaba.kananisarche/Zh/outputs/2025-04-20|09:08:09|RNN_FACENET_5CV"
# data =table_results(
# results_directory_path  = path,
# output_directory_path   = None,
# output_file_path_prefix = 'results',
# cmap                    = "Blues",
# reverse_cmap            = False,
# header_1_bg_color       = 'silver',
# header_1_text_color     = 'black',
# header_2_bg_color       = 'darkgrey',
# header_2_text_color     = 'black',
# maximum_tries           = 1,
# slice_or_video          = 'video', #slice, video
# video_voting_method     = 'class_average', #majority, class_average, average
# height                  = 2,
# width                   = 5,
# )

Try Number 0 Succeeded.


In [ ]:
# data = confusion_matrix_results(
#     results_directory_path  = path,
#     unique_classes          = [2, 6.5, 8.5],
#     output_directory_path   = None,
#     output_file_name_prefix = "results",
#     slice_or_video          = 'slice',     #slice,    video
#     video_voting_method     = 'None',   #majority, class_average, average
#     index                   = -1,          # positive for any index and negative for all
#     split                   = "all",       # train,   val,           test, all
#     cmap                    = "Blues",
#     maximum_tries           = 5,
#     corner_text_color       = "white",
#     corner_bg_color         = "IndianRed",
#     pred_true_text_color    = "black",
#     pred_true_bg_color      = "silver",
#     labels_bg_color         = "beige",
#     labels_text_color       = "black",
#     height                  = 5,
#     width                   = None,
# )

split - train
Try Number 0 Succeeded.
split - test
Try Number 0 Succeeded.
split - val
Try Number 0 Succeeded.


In [1]:
import numpy as np
import pandas as pd
from typing import List, Optional, Tuple
from keras.src.utils import pad_sequences

def reshape_data_rnn(
    df: pd.DataFrame,
    feature_columns: List[str],
    slice_size: int,
    step_size: Optional[int],
    groupby_column: str = "file_name",
    sortby_column: str = "frame",
    target_column: str = "target",
    discard_threshold_ratio: float = 0.6,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Reshape data for RNN input by grouping, slicing, and padding sequences.

    Args:
        df (pd.DataFrame)              : Input dataframe.
        feature_columns (List[str])    : List of feature column names.
        slice_size (int)               : Maximum number of slices in each chunk.
        step_size (int)                : Step size for moving to the next chunk.
        groupby_column (str)           : Column to group by (e.g., video or file name).
        sortby_column (str)            : Column to sort within each group (e.g., frame index).
        target_column (str)            : Column containing target labels.
        discard_threshold_ratio (float): Last slice in each video will be discarded if the number of its frames is less than
                                         discard_threshold_ratio*max_slice and will be post padded with zero and kept otherwise.
    Returns:
        Tuple[np.ndarray, np.ndarray]: Feature sequences (X) and corresponding labels (y).
    """
    step_size = step_size or slice_size
    X, y, g = [], [], []

    for group_identifier, group in df.groupby(groupby_column):
        # Sort by the specified column to maintain temporal order
        group = group.sort_values(by=sortby_column, ascending=True)

        y_group = group[target_column].unique()
        assert (
            len(y_group) == 1
        ), f"Expected one unique target value, but found {len(y_group)}"
        y_group = y_group[0]

        return group, feature_columns

        # Extract features as a list of lists
        slices = group[feature_columns].values.tolist()

        # Create chunks with the specified step size
        for start_idx in range(0, len(slices) - slice_size + 1, step_size):
            chunk = slices[start_idx : start_idx + slice_size]
            # print(chunk)
            X.append(chunk)
            y.append(y_group)
            g.append(group_identifier)

        discard_threshold_frame = int(discard_threshold_ratio * slice_size)
        if len(slices) - (start_idx + slice_size) > discard_threshold_frame:
            chunk = slices[start_idx + slice_size : len(slices)]
            X.append(chunk)
            y.append(y_group)
            g.append(group_identifier)

    # Pad sequences to ensure uniform length
    X_padded = pad_sequences(X, maxlen=slice_size, padding="post", dtype="float32")

    return np.array(X_padded), np.array(y), np.array(g)

2025-04-23 07:06:04.723261: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
%cd '/home/mojtaba.kananisarche/Zh'
from src.utils.load_data import load_data

path_to_data = "/work/long_lab/Mojtaba/data/UTA/1.3.3-preprocessed_merged"
preprocessed_data = load_data(
    input_path=path_to_data,
    # file_names=["facenet_preprocessed_alternativemapping.csv"],
    file_names=["facenet_preprocessed_alternativemapping_croppedonly.csv"],
    nrows=1000,
)
columns_to_drop = [
    "file_name",
    "file_name_1",
    "type",
    "frame",
    "fold",
    "subject_id",
]
groupby_column, sortby_column, target_column = "file_name", "frame", "target"
columns_to_drop_1 = [
    column_name
    for column_name in columns_to_drop
    if column_name not in [groupby_column, sortby_column, target_column]
]
preprocessed_data = preprocessed_data.drop(columns=columns_to_drop_1)

feature_columns = [
    column_name
    for column_name in preprocessed_data.columns
    if column_name not in columns_to_drop + [target_column]
]

2025-04-23 07:06:36,527 - INFO - Concatenated file: facenet_preprocessed_alternativemapping_croppedonly.csv, current data shape: (999, 135)


/home/mojtaba.kananisarche/Zh


In [3]:
preprocessed_data

,file_name,frame,target,f_0,f_1,f_2,f_3,f_4,f_5,f_6,...,f_118,f_119,f_120,f_121,f_122,f_123,f_124,f_125,f_126,f_127
0,1_04_0_30fps_cropped.csv,1,2.0,0.478,-0.379,-1.477,0.302,0.154,1.059,-1.415,...,-0.203,-1.138,2.229,-0.254,0.832,-0.449,-1.163,0.223,0.197,-1.463
1,1_04_0_30fps_cropped.csv,2,2.0,0.544,-0.406,-1.544,0.239,0.127,1.061,-1.398,...,-0.206,-1.177,2.155,-0.115,0.867,-0.517,-1.227,0.294,0.128,-1.510
2,1_04_0_30fps_cropped.csv,3,2.0,0.692,-0.342,-1.480,0.186,0.013,1.143,-1.368,...,-0.195,-1.259,2.100,-0.177,0.983,-0.672,-1.287,0.256,0.116,-1.514
3,1_04_0_30fps_cropped.csv,4,2.0,0.674,-0.349,-1.516,0.165,0.024,1.116,-1.326,...,-0.155,-1.278,2.089,-0.149,0.973,-0.635,-1.356,0.278,0.114,-1.484
4,1_04_0_30fps_cropped.csv,5,2.0,0.713,-0.299,-1.586,0.202,0.050,1.002,-1.356,...,-0.245,-1.188,2.139,-0.079,0.900,-0.525,-1.336,0.353,0.150,-1.367
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
994,1_04_0_30fps_cropped.csv,995,2.0,0.176,-0.087,-0.899,0.611,-0.876,0.808,-1.663,...,-0.809,-0.271,1.886,-0.506,0.760,-0.502,-1.625,1.090,0.249,-1.484
995,1_04_0_30fps_cropped.csv,996,2.0,0.021,-0.404,-0.966,0.555,-0.818,0.817,-1.712,...,-0.791,-0.449,1.652,-0.548,0.922,-0.366,-1.656,1.046,0.387,-1.499
996,1_04_0_30fps_cropped.csv,997,2.0,-0.195,-0.463,-0.912,0.540,-0.789,0.675,-1.642,...,-0.697,-0.564,1.858,-0.500,0.886,-0.525,-1.798,0.950,0.357,-1.364
997,1_04_0_30fps_cropped.csv,998,2.0,-0.273,-0.354,-1.095,0.590,-0.723,0.687,-1.702,...,-0.746,-0.653,1.748,-0.406,0.845,-0.602,-1.719,1.060,0.496,-1.376


In [4]:
g, f = reshape_data_rnn(
    df=preprocessed_data,
    feature_columns=feature_columns,
    slice_size=100,
    step_size=100,
    discard_threshold_ratio=0.7,
    groupby_column=groupby_column,
    sortby_column=sortby_column,
    target_column=target_column,
)

In [5]:
g

,file_name,frame,target,f_0,f_1,f_2,f_3,f_4,f_5,f_6,...,f_118,f_119,f_120,f_121,f_122,f_123,f_124,f_125,f_126,f_127
0,1_04_0_30fps_cropped.csv,1,2.0,0.478,-0.379,-1.477,0.302,0.154,1.059,-1.415,...,-0.203,-1.138,2.229,-0.254,0.832,-0.449,-1.163,0.223,0.197,-1.463
1,1_04_0_30fps_cropped.csv,2,2.0,0.544,-0.406,-1.544,0.239,0.127,1.061,-1.398,...,-0.206,-1.177,2.155,-0.115,0.867,-0.517,-1.227,0.294,0.128,-1.510
2,1_04_0_30fps_cropped.csv,3,2.0,0.692,-0.342,-1.480,0.186,0.013,1.143,-1.368,...,-0.195,-1.259,2.100,-0.177,0.983,-0.672,-1.287,0.256,0.116,-1.514
3,1_04_0_30fps_cropped.csv,4,2.0,0.674,-0.349,-1.516,0.165,0.024,1.116,-1.326,...,-0.155,-1.278,2.089,-0.149,0.973,-0.635,-1.356,0.278,0.114,-1.484
4,1_04_0_30fps_cropped.csv,5,2.0,0.713,-0.299,-1.586,0.202,0.050,1.002,-1.356,...,-0.245,-1.188,2.139,-0.079,0.900,-0.525,-1.336,0.353,0.150,-1.367
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
994,1_04_0_30fps_cropped.csv,995,2.0,0.176,-0.087,-0.899,0.611,-0.876,0.808,-1.663,...,-0.809,-0.271,1.886,-0.506,0.760,-0.502,-1.625,1.090,0.249,-1.484
995,1_04_0_30fps_cropped.csv,996,2.0,0.021,-0.404,-0.966,0.555,-0.818,0.817,-1.712,...,-0.791,-0.449,1.652,-0.548,0.922,-0.366,-1.656,1.046,0.387,-1.499
996,1_04_0_30fps_cropped.csv,997,2.0,-0.195,-0.463,-0.912,0.540,-0.789,0.675,-1.642,...,-0.697,-0.564,1.858,-0.500,0.886,-0.525,-1.798,0.950,0.357,-1.364
997,1_04_0_30fps_cropped.csv,998,2.0,-0.273,-0.354,-1.095,0.590,-0.723,0.687,-1.702,...,-0.746,-0.653,1.748,-0.406,0.845,-0.602,-1.719,1.060,0.496,-1.376


In [6]:
f

['f_0',
 'f_1',
 'f_2',
 'f_3',
 'f_4',
 'f_5',
 'f_6',
 'f_7',
 'f_8',
 'f_9',
 'f_10',
 'f_11',
 'f_12',
 'f_13',
 'f_14',
 'f_15',
 'f_16',
 'f_17',
 'f_18',
 'f_19',
 'f_20',
 'f_21',
 'f_22',
 'f_23',
 'f_24',
 'f_25',
 'f_26',
 'f_27',
 'f_28',
 'f_29',
 'f_30',
 'f_31',
 'f_32',
 'f_33',
 'f_34',
 'f_35',
 'f_36',
 'f_37',
 'f_38',
 'f_39',
 'f_40',
 'f_41',
 'f_42',
 'f_43',
 'f_44',
 'f_45',
 'f_46',
 'f_47',
 'f_48',
 'f_49',
 'f_50',
 'f_51',
 'f_52',
 'f_53',
 'f_54',
 'f_55',
 'f_56',
 'f_57',
 'f_58',
 'f_59',
 'f_60',
 'f_61',
 'f_62',
 'f_63',
 'f_64',
 'f_65',
 'f_66',
 'f_67',
 'f_68',
 'f_69',
 'f_70',
 'f_71',
 'f_72',
 'f_73',
 'f_74',
 'f_75',
 'f_76',
 'f_77',
 'f_78',
 'f_79',
 'f_80',
 'f_81',
 'f_82',
 'f_83',
 'f_84',
 'f_85',
 'f_86',
 'f_87',
 'f_88',
 'f_89',
 'f_90',
 'f_91',
 'f_92',
 'f_93',
 'f_94',
 'f_95',
 'f_96',
 'f_97',
 'f_98',
 'f_99',
 'f_100',
 'f_101',
 'f_102',
 'f_103',
 'f_104',
 'f_105',
 'f_106',
 'f_107',
 'f_108',
 'f_109',
 'f_110',
